In [3]:
import pandas as pd
import pyarrow.parquet as pq
from glob import glob
from tqdm import tqdm
import os

In [26]:
df = pd.read_parquet('../data/input/tennis_data/20240201/data/raw/raw_match_parquet/away_team_11998445.parquet')
print(df.columns)

Index(['match_id', 'name', 'slug', 'gender', 'user_count', 'residence',
       'birthplace', 'height', 'weight', 'plays', 'turned_pro',
       'current_prize', 'total_prize', 'player_id', 'current_rank',
       'name_code', 'country', 'full_name'],
      dtype='object')


In [28]:
df = pd.read_parquet('../data/input/tennis_data/20240201/data/raw/raw_match_parquet/home_team_score_12017473.parquet')
print(df.columns)

Index(['match_id', 'current_score', 'display_score', 'period_1', 'period_2',
       'period_3', 'period_4', 'period_5', 'period_1_tie_break',
       'period_2_tie_break', 'period_3_tie_break', 'period_4_tie_break',
       'period_5_tie_break', 'normal_time'],
      dtype='object')


In [36]:
from tqdm import tqdm
import os
away_files = glob('../data/input/tennis_data/2024*/data/raw/raw_match_parquet/away*.parquet')
home_files = glob('../data/input/tennis_data/2024*/data/raw/raw_match_parquet/home*.parquet')

all_files = away_files + home_files

heights = []

for file in tqdm(all_files):
    try:
        # چک کن ستون height توی فایل هست یا نه
        parquet_file = pq.ParquetFile(file)
        column_names = parquet_file.schema.names
        
        if 'height' in column_names:
            df = pd.read_parquet(file, columns=['height'])
            heights.extend(df['height'].dropna().tolist())
            
    except Exception as e:
        print(f"Error reading file {file}: {e}")

if heights:
    average_height = sum(heights) / len(heights)
    print(f"Average height: {average_height:.2f} cm")
else:
    print("No height data found.")


100%|██████████| 120030/120030 [04:23<00:00, 455.17it/s] 

Average height: 1.82 cm


In [ ]:
away_files = glob('../data/input/tennis_data/2024*/data/raw/raw_match_parquet/away*.parquet')
home_files = glob('../data/input/tennis_data/2024*/data/raw/raw_match_parquet/home*.parquet')
all_files = away_files + home_files

heights = []
ranks = []

for file in tqdm(all_files):
    try:
        # Check if required columns exist
        parquet_file = pq.ParquetFile(file)
        column_names = parquet_file.schema.names

        if 'height' in column_names and 'current_rank' in column_names:
            df = pd.read_parquet(file, columns=['height', 'current_rank'])

            # Drop rows where either is missing
            df = df.dropna(subset=['height', 'current_rank'])

            # Append data
            heights.extend(df['height'].tolist())
            ranks.extend(df['current_rank'].tolist())

    except Exception as e:
        print(f"Error reading file {file}: {e}")

df = pd.DataFrame({
    'height': heights,
    'current_rank': ranks
})

correlation = df["height"].corr(df["current_rank"])
print(f"Correlation between height and current rank: {correlation:.2f}")

100%|██████████| 120030/120030 [37:43<00:00, 53.03it/s]   

Correlation between height and current rank: 0.08


In [7]:
import pandas as pd
import pyarrow.parquet as pq
from glob import glob
from tqdm import tqdm
from collections import defaultdict

# Load all relevant parquet files
event_files = glob('../data/input/tennis_data/2024*/data/raw/raw_match_parquet/event_*.parquet')
home_files = glob('../data/input/tennis_data/2024*/data/raw/raw_match_parquet/home_team_*.parquet')
away_files = glob('../data/input/tennis_data/2024*/data/raw/raw_match_parquet/away_team_*.parquet')

# Step 1: Build match_id -> winner_code map
winner_map = {}
for file in tqdm(event_files, desc="Loading winners"):
    try:
        df = pd.read_parquet(file, columns=['match_id', 'winner_code'])
        for _, row in df.iterrows():
            winner_map[row['match_id']] = row['winner_code']
    except Exception as e:
        print(f"Error reading event file {file}: {e}")

# Step 2: Initialize player stats
player_stats = defaultdict(lambda: {'wins': 0, 'total': 0})

# Step 3: Loop through home and away pairs
for home_file, away_file in tqdm(zip(home_files, away_files), total=len(home_files), desc="Processing matches"):
    try:
        home_df = pd.read_parquet(home_file)
        away_df = pd.read_parquet(away_file)

        # Basic validity check
        if home_df.empty or away_df.empty:
            continue

        home = home_df.iloc[0]
        away = away_df.iloc[0]

        # Ensure required columns exist
        required_columns = ['match_id', 'player_id', 'current_rank']
        if not all(col in home and col in away for col in required_columns):
            continue

        match_id = home['match_id']
        if match_id not in winner_map:
            continue

        winner_code = winner_map[match_id]

        home_id = home['player_id']
        away_id = away['player_id']
        home_rank = home['current_rank']
        away_rank = away['current_rank']

        # Skip if rank info is missing
        if pd.isna(home_rank) or pd.isna(away_rank):
            continue

        # If away is top 10, evaluate home
        if away_rank <= 10:
            player_stats[home_id]['total'] += 1
            if winner_code == 1:
                player_stats[home_id]['wins'] += 1

        # If home is top 10, evaluate away
        if home_rank <= 10:
            player_stats[away_id]['total'] += 1
            if winner_code == 2:
                player_stats[away_id]['wins'] += 1

    except Exception as e:
        print(f"Error processing match {home_file} and {away_file}: {e}")

# Step 4: Calculate win percentages
results = []
for player_id, stats in player_stats.items():
    if stats['total'] > 0:
        win_pct = stats['wins'] / stats['total']
        results.append((player_id, win_pct, stats['wins'], stats['total']))

# Step 5: Display best
if results:
    best = max(results, key=lambda x: x[1])
    print(f"Player ID {best[0]} has the highest winning percentage "
          f"against top 10 opponents: {best[1]*100:.2f}% "
          f"({best[2]} wins out of {best[3]} matches)")
else:
    print("No valid data found to calculate statistics.")


Processing matches:  98%|█████████▊| 59256/60774 [09:41<00:14, 101.90it/s]

Player ID 19957 has the highest winning percentage against top 10 opponents: 100.00% (2 wins out of 2 matches)
